In [1]:
import numpy as np

X = np.load("../processed/X.npy")
y = np.load("../processed/y.npy")
subjects = np.load("../processed/subjects.npy")

print(X.shape)
print(y.shape)
print(subjects.shape)

(4635, 7, 297)
(4635,)
(4635,)


In [2]:
unique_subjects = np.unique(subjects)

print("Number of subjects:", len(unique_subjects))

Number of subjects: 103


In [3]:
from sklearn.model_selection import KFold

subject_kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [4]:
for fold, (train_sub_idx, test_sub_idx) in enumerate(
    subject_kfold.split(unique_subjects)
):

    train_subjects = unique_subjects[train_sub_idx]
    test_subjects = unique_subjects[test_sub_idx]

    print("Fold:", fold + 1)

    print("Train Subjects:", len(train_subjects))
    print("Test Subjects:", len(test_subjects))

    print(
        "Intersection:",
        np.intersect1d(
            train_subjects,
            test_subjects
        )
    )

    break

Fold: 1
Train Subjects: 92
Test Subjects: 11
Intersection: []


In [5]:
train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

In [6]:
X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(4140, 7, 297)
(4140,)
(495, 7, 297)
(495,)


In [7]:
print(
    np.unique(
        subjects[train_mask]
    )[:10]
)

print(
    np.unique(
        subjects[test_mask]
    )[:10]
)

print(
    np.intersect1d(
        np.unique(subjects[train_mask]),
        np.unique(subjects[test_mask])
    )
)

[ 2  3  4  5  6  7  8  9 10 12]
[ 1 11 19 31 41 44 47 49 64 69]
[]


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_flat = X_train.reshape(
    -1,
    297
)

X_test_flat = X_test.reshape(
    -1,
    297
)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

In [9]:
X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

print(X_train.shape)
print(X_test.shape)

(4140, 7, 297)
(495, 7, 297)


In [10]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [11]:
inputs = Input(shape=(7,297))

x = LSTM(
    256,
    return_sequences=True
)(inputs)

x = Dropout(0.2)(x)

x = LSTM(
    256,
    return_sequences=True
)(x)

x = Dropout(0.1)(x)

x = LSTM(
    256,
    return_sequences=False
)(x)

x = Dropout(0.2)(x)

outputs = Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(
    inputs,
    outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 7, 297)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 7, 256)         │       567,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,618,177 (6.17 MB)

 Trainable params: 1,618,177 (6.17 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [13]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 12s 64ms/step - accuracy: 0.5832 - loss: 0.6682 - val_accuracy: 0.6570 - val_loss: 0.6369
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.7203 - loss: 0.5379 - val_accuracy: 0.6618 - val_loss: 0.6169
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 67ms/step - accuracy: 0.8221 - loss: 0.3840 - val_accuracy: 0.6715 - val_loss: 0.7152
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 72ms/step - accuracy: 0.9023 - loss: 0.2416 - val_accuracy: 0.6836 - val_loss: 0.8809
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.9407 - loss: 0.1475 - val_accuracy: 0.6908 - val_loss: 1.0607
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 64ms/step - accuracy: 0.9646 - loss: 0.0943 - val_accuracy: 0.6739 - val_loss: 1.2119
Epoch 7/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.9750 - loss: 0.0714 - val_accuracy: 0.7053 - val_loss: 1.0623


In [15]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

pred = (pred > 0.5).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step
              precision    recall  f1-score   support

           0       0.66      0.69      0.67       254
           1       0.66      0.63      0.64       241

    accuracy                           0.66       495
   macro avg       0.66      0.66      0.66       495
weighted avg       0.66      0.66      0.66       495



In [16]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [17]:
X = np.load("../processed/X.npy")
y = np.load("../processed/y.npy")
subjects = np.load("../processed/subjects.npy")

print(X.shape)
print(y.shape)
print(subjects.shape)

(4635, 7, 297)
(4635,)
(4635,)


In [18]:
unique_subjects = np.unique(subjects)

subject_kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

print("Subjects:", len(unique_subjects))

Subjects: 103


In [19]:
acc_scores = []
prec_scores = []
rec_scores = []

In [20]:
def build_model():

    inputs = Input(shape=(7,297))

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256,
        return_sequences=False
    )(x)

    x = Dropout(0.2)(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [21]:
fold = 1

for train_sub_idx, test_sub_idx in subject_kfold.split(
    unique_subjects
):

    print(f"\n========== Fold {fold} ==========")

    train_subjects = unique_subjects[
        train_sub_idx
    ]

    test_subjects = unique_subjects[
        test_sub_idx
    ]

    train_mask = np.isin(
        subjects,
        train_subjects
    )

    test_mask = np.isin(
        subjects,
        test_subjects
    )

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    print("Train:", X_train.shape)
    print("Test :", X_test.shape)

    scaler = StandardScaler()

    X_train_flat = X_train.reshape(
        -1,
        297
    )

    X_test_flat = X_test.reshape(
        -1,
        297
    )

    X_train_flat = scaler.fit_transform(
        X_train_flat
    )

    X_test_flat = scaler.transform(
        X_test_flat
    )

    X_train = X_train_flat.reshape(
        X_train.shape
    )

    X_test = X_test_flat.reshape(
        X_test.shape
    )

    model = build_model()

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    pred = model.predict(
        X_test,
        verbose=0
    )

    pred = (pred > 0.5).astype(int)

    acc = accuracy_score(
        y_test,
        pred
    )

    prec = precision_score(
        y_test,
        pred
    )

    rec = recall_score(
        y_test,
        pred
    )

    acc_scores.append(acc)
    prec_scores.append(prec)
    rec_scores.append(rec)

    print(
        "Accuracy :",
        round(acc,4)
    )

    print(
        "Precision:",
        round(prec,4)
    )

    print(
        "Recall   :",
        round(rec,4)
    )

    fold += 1


========== Fold 1 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.6949
Precision: 0.6718
Recall   : 0.7303

========== Fold 2 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.6646
Precision: 0.6721
Recall   : 0.656

========== Fold 3 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.7091
Precision: 0.7202
Recall   : 0.6972

========== Fold 4 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.6956
Precision: 0.6533
Recall   : 0.81

========== Fold 5 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.6444
Precision: 0.616
Recall   : 0.733

========== Fold 6 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.6733
Precision: 0.6759
Recall   : 0.6547

========== Fold 7 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.6511
Precision: 0.6772
Recall   : 0.5714

========== Fold 8 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.6578
Precision: 0.6865
Recall 

In [22]:
print("\n========== FINAL ==========")

print(
    "Mean Accuracy:",
    np.mean(acc_scores)
)

print(
    "Mean Precision:",
    np.mean(prec_scores)
)

print(
    "Mean Recall:",
    np.mean(rec_scores)
)


========== FINAL ==========
Mean Accuracy: 0.6850909090909091
Mean Precision: 0.6794435830740754
Mean Recall: 0.6949372485858156
